# 2-D Gaussian Process Adaptive Sampling

This example builds and tunes a vector-valued Gaussian process for a function from a two-dimensional domain to two outputs. We begin with a small set of noisy observations, tune the covariance length scale, then add observations where the posterior variance is largest.

In [ ]:
raise ValueError

import jax

# Un/comment this for double/single precision:
# jax.config.update('jax_enable_x64', True)

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import UncertainSCI.gp as gp


D = 2
C = 2

DOMAIN = jnp.array([[0.0, 0.0], [2.0, 1.0]])
NOISE_VARIANCE = 1e-2

SEED = 0x01234567
GP_SEED = 0xdeadbeef
rng = np.random.default_rng(SEED)

N_PLOT_E1 = 200  # Need not equal below.
N_PLOT_E2 = 100
FIGSIZE = (7, 7 / 1.6)
FIGDPI = None

# Colormap/quadmesh kwargs:
MEAN_KWARGS = dict(vmin=-2.0, vmax=2.0, cmap='Spectral_r', shading='auto')
VARIANCE_KWARGS = dict(vmin=0.0, vmax=1.0, cmap='RdYlGn_r', shading='auto')

def plot_in_subplots(nrows=1, ncols=1, **kwargs):
    """
    Returns ``(fig, axes)`` with plot scaled correctly for ``(nrows, ncols)``.
    """
    if 'figsize' in kwargs:
        raise ValueError("kwargs had 'figsize' key: that's the whole point of this function!")
    return plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * FIGSIZE[0], nrows * FIGSIZE[1]),
        **kwargs
    )


## True function and noisy observations

Our hidden function has two spatially varying outputs. Observations have independent Gaussian noise with a known, constant variance. `GaussianProcess.condition` expects this variance—not its square root.

In [ ]:
x_plot = jnp.stack(
    jnp.meshgrid(
        jnp.linspace(*DOMAIN[:, 0], N_PLOT_E1),
        jnp.linspace(*DOMAIN[:, 1], N_PLOT_E2)
    ),
    axis=-1
)

def f_hidden(x):
    a = 2
    return jnp.stack(
        (
            jnp.sin(a * 2 * jnp.pi * x[..., 0]**2) * x[..., 1],
            jnp.cos(a * 2 * jnp.pi * x[..., 1]**2) * x[..., 0]
        ),
        axis=-1
    )

def f_noisy(x):
    y = f_hidden(x)
    noise = jnp.sqrt(NOISE_VARIANCE) * jnp.asarray(
        rng.normal(size=y.shape)
    )
    return y + noise

Let's plot this function:

In [ ]:
truth = f_hidden(x_plot)
noisy = f_noisy(x_plot)

for output in range(C):
    fig, axes = plot_in_subplots(1, 2, dpi=FIGDPI)
    for ax, values, title in zip(
        axes,
        (truth[..., output], noisy[..., output]),
        ('True Function', 'Noisy Realization')
    ):
        plotted = ax.pcolormesh(x_plot[..., 0], x_plot[..., 1], values, **MEAN_KWARGS)
        fig.colorbar(plotted, ax=ax)
        ax.set_aspect('equal')
        ax.set_xlabel('$x_1$')
        ax.set_ylabel('$x_2$')
        ax.set_title(f'{title}, Output {output + 1}')
    fig.tight_layout()
    plt.show()


## Define the Gaussian process

The prior has a fixed zero affine mean. A Gaussian coordinate kernel models spatial correlation, while the Kronecker output matrix models correlation between the two outputs. Both covariance matrices are tuned from the observations.

In [ ]:
mu = gp.mean.Affine(
    dim=D,
    cdim=C,
    a=jnp.zeros((C, D)),
    b=jnp.zeros(C),
    a_is_static=True,
    b_is_static=True
)
coordinate_kernel = gp.kernel.Gaussian(
    dim=D,
    cdim=1,
    D=jnp.array([[1.0, 0.25], [0.25, 2.0]])
)
k = gp.kernel.Kronecker(
    dim=D,
    cdim=C,
    k=coordinate_kernel,
    C=jnp.array([[1.0, 0.2], [0.2, 1.0]])
)
g = gp.GaussianProcess(
    dim=D,
    cdim=C,
    mu=mu,
    k=k,
    nugget=1e-4,
    seed=GP_SEED
)

## Condition and tune

Begin with ten randomly located observations. Because the noise variance is shared by every observation and output, it can be passed as a scalar and the Kronecker covariance structure remains available.

In [ ]:
N_INITIAL = 10
train_x = DOMAIN[0] + (DOMAIN[1] - DOMAIN[0]) * jnp.asarray(
    rng.random((N_INITIAL, D))
)
train_y = f_noisy(train_x)

g.condition(train_x, train_y, NOISE_VARIANCE)
losses = g.tune(max_iter=250, rel=1e-3, window=20, patience=2)

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(losses)
ax.set_xlabel('Optimization Step')
ax.set_ylabel('Negative Log Marginal Likelihood')
ax.set_title('Initial Hyperparameter Tuning')
fig.tight_layout()
plt.show()


The initial posterior is shown as a pair of plots for each output. Keeping each figure to the posterior mean and marginal variance makes the spatial relationship easier to read.

In [ ]:
import matplotlib.axes as mpl_axes
import numpy.typing as npt

def plot_distribution_mean_2d(
    ax: mpl_axes.Axes,
    g: gp.GaussianProcess,
    x: jax.Array | npt.NDArray,
    channel: int,
    which: str,
    **kwargs
):
    if x.ndim != 3:
        raise ValueError
    n1, n2, d = x.shape

    if d != g.dim:
        raise ValueError

    x_flat = x.reshape((-1, d))
    if which == 'prior':
        y_flat = g.prior_mean(x_flat)
    elif which == 'posterior':
        y_flat = g.posterior_mean(x_flat)
    else:
        raise ValueError
    y = y_flat.reshape((n1, n2, -1))

    p = ax.pcolormesh(
        x[..., 0],
        x[..., 1],
        y[..., channel],
        **kwargs
    )
    plt.colorbar(
        p,
        ax=ax
    )

def plot_distribution_variance_2d():
    pass

def plot_realization_2d(
    ax: mpl_axes.Axes,
    g: gp.GaussianProcess,
    x: jax.Array | npt.NDArray,
    channel: int,
    which: str,
    **kwargs
):
    if x.ndim != 3:
        raise ValueError
    n1, n2, d = x.shape

    if d != g.dim:
        raise ValueError

    x_flat = x.reshape((-1, d))
    if which == 'prior':
        y_flat = g.prior_realization(x_flat)
    elif which == 'posterior':
        y_flat = g.posterior_realization(x_flat)
    else:
        raise ValueError
    y = y_flat.reshape((n1, n2, -1))

    p = ax.pcolormesh(
        x[..., 0],
        x[..., 1],
        y[..., channel],
        **kwargs
    )
    plt.colorbar(
        p,
        ax=ax
    )

fig, (ax1, ax2) = plot_in_subplots(1, 2)

plot_distribution_mean_2d(
    ax1,
    g,
    x_plot,
    0,
    'prior',
    **MEAN_KWARGS
)

plot_realization_2d(
    ax1,
    g,
    x_plot,
    0,
    'prior',
    **MEAN_KWARGS
)

# for output in range(C):
#     fig, axes = plt.subplots(1, 2, figsize=(2 * FIGSIZE[0], FIGSIZE[1]))
#     mean_plot = gp.vis.plot_distribution_mean_2d(
#         axes[0], g, (x_plot[..., 0], x_plot[..., 1]), output=output, **MEAN_KWARGS
#     )
#     variance_plot = gp.vis.plot_distribution_variance_2d(
#         axes[1], g, (x_plot[..., 0], x_plot[..., 1]), output=output, **VARIANCE_KWARGS
#     )
#     fig.colorbar(mean_plot, ax=axes[0])
#     fig.colorbar(variance_plot, ax=axes[1])
#     for ax in axes:
#         ax.set_xlabel('$x_1$')
#         ax.set_ylabel('$x_2$')
#     fig.tight_layout()
#     plt.show()


## Adaptively add observations

At each iteration, we find the mesh point with the largest total marginal variance across both outputs. Starting there, `get_sample_point` refines the coordinate by directly maximizing posterior variance within the rectangular domain. The new observation is added and the hyperparameters are tuned again.

In [ ]:
N_ADAPTIVE = 8

for iteration in range(N_ADAPTIVE):
    covariance = g.posterior_covariance(x_plot, x_plot)
    marginal_variance = jnp.diag(covariance).reshape((-1, C))
    x_start = x_plot[jnp.argmax(jnp.sum(marginal_variance, axis=1))]
    x_new = g.get_sample_point(
        x_start,
        ranges=DOMAIN,
        n=100,
        optim_kwargs={'learning_rate': 1e-2}
    )

    train_x = jnp.concatenate((train_x, x_new), axis=0)
    train_y = jnp.concatenate((train_y, f_noisy(x_new)), axis=0)
    g.condition(train_x, train_y, NOISE_VARIANCE)
    losses = g.tune(max_iter=100, rel=1e-3, window=10, patience=2)

    print(
        f'Iteration {iteration + 1}: sampled '
        f'({float(x_new[0, 0]):.3f}, {float(x_new[0, 1]):.3f})'
    )

## Final posterior

The diamonds mark earlier observations and the larger red circle marks the final adaptively selected coordinate. Compared with the initial posterior, uncertainty has been reduced across the domain.

In [ ]:
for output in range(C):
    fig, axes = plt.subplots(1, 2, figsize=(2 * FIGSIZE[0], FIGSIZE[1]))
    mean_plot = gp.vis.plot_distribution_mean_2d(
        axes[0],
        g,
        mesh,
        output=output,
        colorlast=True,
        **MEAN_KWARGS
    )
    variance_plot = gp.vis.plot_distribution_variance_2d(
        axes[1],
        g,
        mesh,
        output=output,
        colorlast=True,
        **VARIANCE_KWARGS
    )
    fig.colorbar(mean_plot, ax=axes[0])
    fig.colorbar(variance_plot, ax=axes[1])
    for ax in axes:
        ax.set_xlabel('$x_1$')
        ax.set_ylabel('$x_2$')
    fig.tight_layout()
    plt.show()